In [2]:
import glob

import numpy as np
import pandas as pd
from model.ctabgan import CTABGAN
from model.eval.evaluation import get_utility_metrics, privacy_metrics, stat_sim

In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1.9.1+cu111
True
1
NVIDIA A100 80GB PCIe


In [4]:
num_exp = 5
dataset = "MimicIV"
real_path = "Real_Datasets/MimicIV_clean.csv"
fake_file_root = "Fake_Datasets"

In [5]:
all_discriminator_snapshots = {}

for i in range(num_exp):

    print(f"Training run {i}")
    
    synthesizer = CTABGAN(
        raw_csv_path=real_path,
        test_ratio=0.20,
        categorical_columns=["gender", "hospital_expire_flag"],
        log_columns=[],  # none unless you want to log-transform skewed labs
        mixed_columns={},  # none unless you confirm heavy zero-mass columns
        general_columns=[
            "heart_rate_min",
            "heart_rate_max",
            "heart_rate_mean",
            "sbp_min",
            "sbp_max",
            "sbp_mean",
            "dbp_min",
            "dbp_max",
            "dbp_mean",
            "resp_rate_min",
            "resp_rate_max",
            "resp_rate_mean",
            "temperature_min",
            "temperature_max",
            "temperature_mean",
            "spo2_min",
            "spo2_max",
            "spo2_mean",
            "creatinine_min",
            "creatinine_max",
            "sodium_min",
            "sodium_max",
            "potassium_min",
            "potassium_max",
            "hemoglobin_min",
            "hemoglobin_max",
            "wbc_min",
            "wbc_max",
        ],
        non_categorical_columns=[],  # since we dropped IDs
        integer_columns=["age_at_intime"],
        problem_type={"Classification": "hospital_expire_flag"},
        class_dim=(256, 256),
        random_dim=100,
        num_channels=64,
        l2scale=1e-5,
        batch_size=512,
        epochs=150,
    )

    discriminator_snapshots = synthesizer.fit()

    all_discriminator_snapshots[f"run_{i}"] = discriminator_snapshots
    
    syn = synthesizer.generate_samples()
    syn.to_csv(
        fake_file_root
        + "/"
        + dataset
        + "/"
        + dataset
        + "_fake_{exp}.csv".format(exp=i),
        index=False,
    )

dside = synthesizer.synthesizer.dside
layers_D = synthesizer.synthesizer.layers_D

100%|██████████| 150/150 [08:21<00:00,  3.35s/it]


Finished training in 506.799156665802  seconds.


100%|██████████| 150/150 [08:17<00:00,  3.32s/it]


Finished training in 499.9102990627289  seconds.


100%|██████████| 150/150 [08:16<00:00,  3.31s/it]


Finished training in 498.73297119140625  seconds.


100%|██████████| 150/150 [08:17<00:00,  3.32s/it]


Finished training in 499.8964970111847  seconds.


100%|██████████| 150/150 [08:15<00:00,  3.30s/it]


Finished training in 497.55955839157104  seconds.


In [ ]:
fake_paths = glob.glob(fake_file_root + "/" + dataset + "/" + "*")

In [ ]:
def make_numeric_for_utility(path_in, path_out, target="hospital_expire_flag"):
    df = pd.read_csv(path_in)

    # Ensure target is last or separated (depending on your eval code)
    y = df[target]
    X = df.drop(columns=[target])

    # One-hot encode any object/categorical columns (e.g., gender)
    X_enc = pd.get_dummies(X, drop_first=False)

    out = pd.concat([X_enc, y], axis=1)
    out.to_csv(path_out, index=False)
    return path_out


real_path_u = make_numeric_for_utility(real_path, "Real_Datasets/MimicIVU.csv")

fake_paths_u = []
for i, fp in enumerate(fake_paths):
    fake_paths_u.append(
        make_numeric_for_utility(fp, f"Fake_Datasets/MimicIV/MimicIV_fakeU_{i}.csv")
    )

    # after generating real one-hot columns, align fake to real:
real_cols = pd.read_csv(real_path_u).columns


def align_to_real(path_in, real_cols, target):
    df = pd.read_csv(path_in)
    y = df[target]
    X = pd.get_dummies(df.drop(columns=[target]), drop_first=False)

    X = X.reindex(columns=[c for c in real_cols if c != target], fill_value=0)
    out = pd.concat([X, y], axis=1)
    return out

In [ ]:
import pandas as pd

df = pd.read_csv(fake_file_root + "/" + dataset + "/" + dataset + "_fakeU_0.csv")
df.info()
df.head()
df.tail()
df.isna().sum()
df.replace([np.inf, -np.inf], np.nan).isna().sum()

In [ ]:
model_dict = {"Classification": ["lr", "dt", "rf", "mlp", "svm"]}
result_mat = get_utility_metrics(
    real_path, fake_paths, "MinMax", model_dict, test_ratio=0.20
)

result_df = pd.DataFrame(result_mat, columns=["Acc", "AUC", "F1_Score"])
result_df.index = list(model_dict.values())[0]
result_df

In [ ]:
mimic_categorical = ["hospital_expire_flag", "gender"]
stat_res_avg = []
for fake_path in fake_paths:
    stat_res = stat_sim(real_path, fake_path, mimic_categorical)
    stat_res_avg.append(stat_res)

stat_columns = [
    "Average WD (Continuous Columns",
    "Average JSD (Categorical Columns)",
    "Correlation Distance",
]
stat_results = pd.DataFrame(
    np.array(stat_res_avg).mean(axis=0).reshape(1, 3), columns=stat_columns
)
stat_results

In [ ]:
priv_res_avg = []
for fake_path in fake_paths_u:
    priv_res = privacy_metrics(real_path_u, fake_path)
    priv_res_avg.append(priv_res)

privacy_columns = [
    "DCR between Real and Fake (5th perc)",
    "DCR within Real(5th perc)",
    "DCR within Fake (5th perc)",
    "NNDR between Real and Fake (5th perc)",
    "NNDR within Real (5th perc)",
    "NNDR within Fake (5th perc)",
]
privacy_results = pd.DataFrame(
    np.array(priv_res_avg).mean(axis=0).reshape(1, 6), columns=privacy_columns
)
privacy_results

In [ ]:
import os

import pandas as pd
from Evaluation.cdf_tail_metrics import CDFTailMetrics
from Evaluation.rare_event_recall import RareEventRecall
from Evaluation.support_coverage import SupportCoverage

n_experiments = 5

cdf_eval = CDFTailMetrics(label_col="hospital_expire_flag", tau=0.9)
sc_eval = SupportCoverage(
    label_col="hospital_expire_flag",
    k=2,  # pairwise support coverage
    n_bins=10,
    rare_threshold=0.05,
    max_subsets=100,  # sample up to 100 feature pairs
    include_label_in_combo=True,
    random_state=42,
)
rer_eval = RareEventRecall(label_col="hospital_expire_flag")

rows = []

for exp in range(n_experiments):
    syn_path = os.path.join(fake_file_root, dataset, f"{dataset}_fakeU_{exp}.csv")
    if not os.path.exists(syn_path):
        print("Missing:", syn_path)
        continue

    res = {"exp": exp, "syn_path": syn_path}
    res.update(cdf_eval.evaluate_paths(real_path_u, syn_path))
    res.update(sc_eval.evaluate_paths(real_path_u, syn_path))
    res.update(rer_eval.evaluate_paths(real_path_u, syn_path))
    rows.append(res)

results_df = pd.DataFrame(rows)
results_df.to_csv("artifacts/MimicEval.csv")
results_df

In [ ]:
from Evaluation.detector_treeshap import DetectorTreeSHAP

xai_detector = DetectorTreeSHAP(
    test_size=0.3,
    random_state=42,
    n_estimators=300,
    output_dir=f"artifacts/xai_detector/{dataset}",
)

detector_results = xai_detector.evaluate_paths(real_path, syn_path)
detector_results

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def reconstruct_discriminator(snapshot, dside, layers_D, device):
    disc = Discriminator(dside, layers_D).to(device)
    disc.load_state_dict(snapshot["state_dict"])
    disc.eval()
    return disc

In [ ]:
reconstructed_discriminators = {}

for run_name, snapshots in all_discriminator_snapshots.items():
    reconstructed_discriminators[run_name] = {}

    for snapshot in snapshots:
        epoch = snapshot["epoch"]

        disc = reconstruct_discriminator(
            snapshot=snapshot,
            dside=dside,
            layers_D=layers_D,
            device=device
        )

        reconstructed_discriminators[run_name][epoch] = disc


In [ ]:
disc_run0_epoch25 = reconstructed_discriminators["run_0"][25]
disc_run1_epoch100 = reconstructed_discriminators["run_1"][100]